# Extração de Dados Complementares - Censo 2022 e CEMPRE

Este notebook complementa as extrações buscando variáveis ausentes listadas:
1. **Alfabetização (Censo 2022)** (Tabela 9543)
2. **Saneamento Básico (Censo 2022 proxy)** (Tabela 9514)
3. **População Urbana (Censo 2010 proxy)** (Tabela 1378 - Filtro Exclusivo Urbano)
4. **Número de Empresas (CEMPRE 2021)** por UF para contornar Erro 500 na Tabela 6442.

In [1]:
import pandas as pd
import requests
import time

def extract_sidra(table_code, period, variables='all', level='N6', query_append=''):
    url = f"https://servicodados.ibge.gov.br/api/v3/agregados/{table_code}/periodos/{period}/variaveis/{variables}?localidades={level}[all]{query_append}"
    response = requests.get(url)
    if response.status_code != 200:
        print(f"Erro na tabela {table_code}: HTTP {response.status_code}")
        return pd.DataFrame()
    data = response.json()
    records = []
    for var_info in data:
        var_name = var_info['variavel']
        for res in var_info['resultados']:
            for serie in res['series']:
                loc_id = serie['localidade']['id']
                loc_name = serie['localidade']['nome']
                valor = serie['serie'].get(str(period))
                records.append({'cod_municipio': loc_id, 'municipio': loc_name, 'variavel': var_name, 'valor': valor})
    df = pd.DataFrame(records)
    if df.empty: return df
    df['valor'] = pd.to_numeric(df['valor'], errors='coerce')
    return df.pivot_table(index=['cod_municipio', 'municipio'], columns='variavel', values='valor', aggfunc='first').reset_index()

In [2]:
print("---- 1. Taxa de Alfabetização (Tabela 9543) ----")
df_alfab = extract_sidra('9543', '2022', '2513')
if not df_alfab.empty:
    df_alfab.columns.name = None
    df_alfab = df_alfab.rename(columns={'Taxa de alfabetização das pessoas de 15 anos ou mais de idade': 'taxa_alfabetizacao'})
    display(df_alfab.head(2))

---- 1. Taxa de Alfabetização (Tabela 9543) ----


,cod_municipio,municipio,taxa_alfabetizacao
0,1100015,Alta Floresta D'Oeste - RO,91.59
1,1100023,Ariquemes - RO,94.08


In [3]:
print("---- 2. População Urbana (Tabela 1378 - Censo 2010 proxy) ----")
df_urbana = extract_sidra('1378', '2010', '93', query_append='&classificacao=1[1]')
if not df_urbana.empty:
    df_urbana.columns.name = None
    df_urbana = df_urbana.rename(columns={'População residente': 'populacao_urbana_2010'})
    display(df_urbana.head(2))

---- 2. População Urbana (Tabela 1378 - Censo 2010 proxy) ----


,cod_municipio,municipio,populacao_urbana_2010
0,1100015,Alta Floresta D'Oeste - RO,13970
1,1100023,Ariquemes - RO,76525


In [4]:
print("---- 3. Saneamento Básico (Tabela 9514 - Censo 2022 proxy) ----")
# Buscamos totais simples pois a granularidade por tipo de rede no SIDRA pode ser complexa via pivot
df_san = extract_sidra('9514', '2022', '12154')
if not df_san.empty:
    df_san.columns.name = None
    df_san = df_san.rename(columns={'Pessoas residentes em domicílios particulares permanentes ocupados': 'pessoas_com_saneamento'})
    display(df_san.head(2))

---- 3. Saneamento Básico (Tabela 9514 - Censo 2022 proxy) ----
Erro na tabela 9514: HTTP 500


In [5]:
print("---- 4. Número de Empresas (Tabela 6442 - CEMPRE 2021) ----")
ufs = [11,12,13,14,15,16,17,21,22,23,24,25,26,27,28,29,31,32,33,35,41,42,43,50,51,52,53]
dfs_cempre = []
for uf in ufs:
    # Extraindo por estado para evitar erro 500
    url = f"https://servicodados.ibge.gov.br/api/v3/agregados/6442/periodos/2021/variaveis/1006?localidades=N6[in n3[{uf}]]"
    res = requests.get(url)
    if res.status_code == 200:
        data = res.json()
        records = []
        for var_info in data:
            for res_obj in var_info['resultados']:
                for serie in res_obj['series']:
                    records.append({
                        'cod_municipio': serie['localidade']['id'],
                        'num_empresas': pd.to_numeric(serie['serie'].get('2021'), errors='coerce')
                    })
        dfs_cempre.append(pd.DataFrame(records))
    time.sleep(0.5)
if dfs_cempre:
    df_cempre = pd.concat(dfs_cempre, ignore_index=True)
    display(df_cempre.head(2))
else:
    df_cempre = pd.DataFrame()

---- 4. Número de Empresas (Tabela 6442 - CEMPRE 2021) ----


In [6]:
print("---- Merge dos Dataframes Complementares ----")
df_complementar = pd.DataFrame(columns=['cod_municipio'])
for df_parcial in [df_alfab, df_urbana, df_san, df_cempre]:
    if not df_parcial.empty:
        if df_complementar.empty:
            df_complementar = df_parcial.copy()
        else:
            df_complementar = pd.merge(df_complementar, df_parcial.drop(columns=['municipio'], errors='ignore'), on='cod_municipio', how='outer')

df_complementar.to_csv('data/censo_2022.csv', index=False)
print(f"Salvo em 'data/censo_2022.csv' com shape: {df_complementar.shape}")
display(df_complementar.head())

---- Merge dos Dataframes Complementares ----
Salvo em 'data/censo_2022.csv' com shape: (5570, 4)


,cod_municipio,municipio,taxa_alfabetizacao,populacao_urbana_2010
0,1100015,Alta Floresta D'Oeste - RO,91.59,13970.0
1,1100023,Ariquemes - RO,94.08,76525.0
2,1100031,Cabixi - RO,89.82,2693.0
3,1100049,Cacoal - RO,93.71,61921.0
4,1100056,Cerejeiras - RO,92.15,14419.0
